# 07 — Reduced study: coverage, paired results and paper export

The primary study has **39 neural trainings**: 18 smooth 2D, 12 nonlinear 1D and 9 nominal brass fits. The ten completed synthetic runs in your archive belong to this plan and are reused. The four optional dimensional demonstrations are outside this budget.

This notebook performs **no training**. Run all cells after notebooks 03–05, or at any time to see what is missing. Fresh reports go to `results/reduced_report/`; the uploaded run files and earlier reports remain unchanged.

A completed training remains a candidate result until numerical checks and scientific interpretation have been reviewed. Missing, failed and blocked jobs stay visible.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "inverse_study").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Extract the complete heat folder and open this notebook from it.")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from inverse_study.common import RESULTS, read_json, write_json, sha256_file, metadata_hash
from inverse_study import completion_report as report
plt.rcParams.update({"font.size": 11, "figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.2, "savefig.bbox": "tight"})
print("Study folder:", ROOT)

In [ ]:
snapshot = report.build_report()
tables, summary, report_dir = snapshot["files"], snapshot["summary"], snapshot["report_dir"]
coverage, brass_coverage = tables["synthetic_coverage"], tables["brass_coverage"]
individual, classical = tables["synthetic_individual_results"], tables["classical_unique_fits"]
print("Declared neural trainings:", summary["synthetic_planned"] + summary["brass_planned"])
print("Completed and reusable:", summary["synthetic_completed"] + summary["brass_completed"])
print("Missing or resumable:", summary["synthetic_missing_or_resumable"] + summary["brass_missing_or_resumable"])
print("Blocked:", summary["synthetic_blocked"] + summary["brass_blocked"])
print("Retained linear classical fits:", summary["retained_classical_unique_fits"])
print("Earlier extra fits preserved outside the primary cohort:", summary["out_of_cohort_candidates_preserved"])
print("Fresh reports:", report_dir)

## 1. Coverage of the exact 39-job plan

`completed` means the data, source, configuration, budget and required saved artifacts match the declared job. A weak reconstruction is still a completed outcome; its error does not trigger another seed. `missing` and `resumable` are eligible for the training queues. `blocked` requires inspecting the mismatch.

Noisy synthetic repetitions are the fixed pairs `(3100, 2026)`, `(3101, 2027)` and `(3102, 2028)`, shared across methods. Clean and brass repetitions use training seeds 2026–2028. Brass uses one physical experiment.

In [ ]:
cols = ["case", "method", "training_seed", "status", "reason", "run_folder"]
combined_coverage = pd.concat([coverage[cols], brass_coverage[cols]], ignore_index=True)
counts = pd.crosstab([combined_coverage.case, combined_coverage.method], combined_coverage.status)
counts["planned"] = counts.sum(axis=1)
display(counts)
display(combined_coverage.loc[~combined_coverage.status.eq("completed"), cols[:-1]])
statuses = tables["all_production_training_statuses"]
if len(statuses):
    failures = statuses[statuses.status.isin(["failed", "interrupted", "unreadable_metadata"])]
    if len(failures):
        print("All saved failures/interruption records, including optional jobs:")
        display(failures)
blocked = tables["blocked_candidate_records"]
if len(blocked): display(blocked)

## 2. Individual outcomes and descriptive variability

Each condition has three planned repetitions. Report all outcomes, median/range and, where meaningful, mean/sample standard deviation. With one available result the standard deviation stays missing; no error bar is invented.

Noisy repetitions combine noise and training variability and cannot separate their variance components. This compact design uses **no crossed bootstrap, significance test or precise confidence claim**. State the effective sample size and restrict conclusions to tested conditions.

In [ ]:
cols = [c for c in ["case", "noise_level", "repetition", "method", "status", "initial_relative_l2",
    "independent_observation_residual", "initial_boundary_linf", "rmse_over_true_jump",
    "jump_height_relative_error", "training_wall_time_s"] if c in individual]
display(individual[cols])
stats = tables["synthetic_descriptive_summary"]
if len(stats):
    display(stats[stats.metric.isin(["initial_relative_l2", "independent_observation_residual", "initial_boundary_linf"])])

## 3. Paired method differences

Compare the same declared repetition for both methods. Missing pairs remain explicit. A negative error difference `a_minus_b` favors method A in that repetition; this alone is not statistical superiority. Retain all three repetitions even when outcomes disagree. Fixed and adaptive Fourier controls remain distinct.

In [ ]:
display(tables["paired_neural_differences"])
display(tables["paired_difference_summary"])

## 4. Classical baselines: count each observation dataset once

The reduced linear study needs eight deterministic fits: Tikhonov and TSVD on one clean and three noisy datasets. All eight are in your archive. Four extra fits on noise seeds 3103 and 3104 remain saved outside the paired primary cohort.

The clean classical fit can accompany three neural initializations, but remains one baseline fit. Notebook 04 supplies the nonlinear comparator for its four observation cases. Parameter trials, solver refinement and neural repetitions are different quantities.

In [ ]:
cols = [c for c in ["case", "noise_level", "noise_seed", "method", "initial_relative_l2",
    "independent_observation_residual", "selection_status", "selected_parameter", "dataset_id"] if c in classical]
display(classical[cols] if len(classical) else pd.DataFrame({"status":["Linear baselines pending."]}))
display(tables["classical_descriptive_summary"])
display(tables["neural_classical_paired_differences"])
display(tables["nonlinear_baseline_coverage"])
nonlinear_baselines = tables["nonlinear_baseline_candidates"]
display(nonlinear_baselines if len(nonlinear_baselines) else pd.DataFrame({"status":["Nonlinear baseline work remains in notebook 04."]}))
extras = tables["preserved_outside_primary_cohort"]
if len(extras):
    display(extras[[c for c in ["case", "method", "noise_level", "noise_seed", "cohort_note"] if c in extras]])
if len(tables["nonlinear_baseline_variants_preserved"]):
    display(tables["nonlinear_baseline_variants_preserved"])

## 5. Independent replay and refinement

Uploaded `evaluation.json` records remain unchanged. Their `forward_replay_requires_refinement` field describes the original check. Subsequent checks of those saved weights appear separately below and require no retraining.

Review terminal residuals, network-versus-independent predictions and initial boundary mismatch. In the jump case also review transition width, jump/plateau errors and oscillations. Numerical refinement does not automatically resolve reconstruction or physical-model errors.

In [ ]:
refinement = tables["independent_replay_refinement"]
display(refinement if len(refinement) else pd.DataFrame({"status":["Run cached replay refinement in notebooks 03 and 04."]}))
reference_path = RESULTS / "verification/nonlinear_reference_check.json"
if reference_path.is_file():
    check = read_json(reference_path)
    display(pd.Series({k: check.get(k) for k in ["passed", "meshes", "relative_acceptance", "sensor_difference_l2", "convergence_scope", "reference_sha256"]}).to_frame("frozen_reference"))

## 6. Candidate figures from actual retained results

Points are actual completed outcomes; labels give available sample sizes. Classical points count unique observation datasets. A logarithmic error axis shows the small classical errors. Missing results are never filled in. These figures replace manuscript figures only after numerical and interpretation checks.

In [ ]:
parts = []
if "initial_relative_l2" in individual: parts.append(individual[individual.status.eq("completed")].copy())
if len(classical): parts.append(classical.copy())
plot_data = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
if len(plot_data):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
    methods = ["plain", "fourier_fixed", "fourier_adaptive", "tikhonov", "tsvd"]
    rng = np.random.default_rng(901)
    for ax, rho in zip(axes, [0.0, 0.05]):
        group = plot_data[(plot_data.case == "smooth2d") & np.isclose(plot_data.noise_level, rho)]
        labels = []
        for i, method in enumerate(methods):
            values = pd.to_numeric(group.loc[group.method.eq(method), "initial_relative_l2"], errors="coerce").to_numpy()
            values = values[np.isfinite(values) & (values > 0)]
            labels.append(f"{method}\nN={len(values)}")
            if len(values):
                ax.scatter(i+rng.uniform(-.09, .09, len(values)), values, s=28, alpha=.8)
                ax.plot([i-.16, i+.16], [np.median(values)]*2, "k-", lw=1.6)
        ax.set_xticks(range(len(methods)), labels, rotation=20, ha="right")
        ax.set_yscale("log")
        ax.set(ylabel="Initial relative L2 error", title=f"Smooth 2D: {rho:.0%} noise")
    fig.savefig(report_dir/"compact_smooth_comparison.svg")
    fig.savefig(report_dir/"compact_smooth_comparison.png", dpi=250)
    plt.show()
else:
    print("No completed results for a figure.")

## 7. Brass: nine nominal fits, one physical experiment

Evaluate withheld times and independent replay for every retained model. Report errors in kelvin and state normalization separately. Repeated neural fits do not create independent physical measurements. Calibration dependence and digitization uncertainty remain source questions.

In [ ]:
display(tables["brass_evidence_coverage"])
brass_results = tables["brass_physical_units"]
display(brass_results if len(brass_results) else pd.DataFrame({"status":["Nine nominal brass fits and assessments are queued in notebook 05."]}))
if len(tables["brass_initial_descriptive_summary"]): display(tables["brass_initial_descriptive_summary"])
if len(tables["brass_independent_replay"]): display(tables["brass_independent_replay"])
if len(tables["brass_postprocessing_failure_history"]):
    print("Cumulative brass assessment/replay failure history:")
    display(tables["brass_postprocessing_failure_history"])

## 8. Evidence review and optional paper export

Keep `FREEZE_FOR_PAPER=False` during computation; it is unnecessary for running or resuming experiments. Every publication gate requires a real evidence file and reviewed interpretation. Training completion is not publication approval.

The export requires all 39 trainings, their assessments and the gates below. Optional dimensional demonstrations remain outside this count; qualify or remove unsupported dimensional claims. A prospective protocol amendment must be documented before changing the cohort.

In [ ]:
gate_path = report_dir/"publication_gates.json"
if not gate_path.exists():
    gates = {name:{"passed":False,"evidence":None} for name in [
      "reference_and_replay_converged", "classical_basis_and_parameter_validation",
      "QR_retained_with_validation_or_explicitly_replaced", "paired_controls_all_outcomes_and_failures_reviewed",
      "small_sample_limits_and_claim_scope_disclosed", "brass_calibration_and_digitization_disclosed",
      "timing_scopes_and_hardware_disclosed", "authors_funding_and_data_release_complete"]}
    write_json(gate_path,gates)
gates=read_json(gate_path)
display(pd.DataFrame(gates).T)

FREEZE_FOR_PAPER = False
if FREEZE_FOR_PAPER:
    if not coverage.status.eq("completed").all() or not brass_coverage.status.eq("completed").all():
        raise RuntimeError("The 39-job plan is incomplete or blocked.")
    if "initial_relative_l2" not in individual or not np.isfinite(individual.initial_relative_l2).all():
        raise RuntimeError("Synthetic assessments are missing or nonfinite.")
    if not tables["nonlinear_baseline_coverage"].status.eq("completed").all():
        raise RuntimeError("The four declared nonlinear classical comparison cases are incomplete.")
    needed=["withheld_metrics_present","replay_metrics_present","refinement_record_present"]
    if not tables["brass_evidence_coverage"][needed].all().all():
        raise RuntimeError("Brass assessment/replay/refinement records are incomplete.")
    for name,gate in gates.items():
        if not gate["passed"] or not gate["evidence"]: raise RuntimeError(f"Unresolved evidence gate: {name}")
        evidence=Path(gate["evidence"])
        if not evidence.is_absolute(): evidence=ROOT/evidence
        if not evidence.is_file(): raise FileNotFoundError(evidence)
    import shutil
    items=sorted(p for p in report_dir.iterdir() if p.is_file())
    files=[{"file":p.name,"sha256":sha256_file(p)} for p in items]
    freeze_id=metadata_hash(files)[:16]
    destination=RESULTS/"paper_freezes"/freeze_id
    destination.mkdir(parents=True,exist_ok=False)
    for p in items: shutil.copy2(p,destination/p.name)
    write_json(destination/"MANIFEST.json",{"freeze_id":freeze_id,"files":files,"design":"compact_paired_v1",
        "instruction":"Use audited rows with their stated limitations when updating the manuscript."})
    print("Paper export:",destination)
else:
    print("Paper freeze is OFF. Actual results and the reduced plan are exported for review.")